<div style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:11px;letter-spacing:.24em;text-transform:uppercase;color:#A20025;margin:0 0 12px;">NeurIPS 2026 Education Track&nbsp;&middot;&nbsp;Generate, then verify</div>

<h1 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:33px;line-height:1.12;margin:0 0 16px;color:#1A1416;letter-spacing:-.01em;">Worked solutions 4 to 7. Molecules, and the definitions behind their checks</h1>

<hr style="border:0;border-top:1px solid #A20025;opacity:.55;margin:0 0 4px;">

<em style="color:#6B6461;font-size:14.5px;">Full working for X4 to X7. The X5 discussion is long on purpose; it is the lesson the rest of the resource is arranged around.</em>

Four tasks, and the middle one carries most of the weight. X4 and X6 have short
answers and long implications. X5 has a short answer and an implication that should
change how you read a results table for good. X7 asks you to put the other three in
order.

If you are marking this rather than learning from it, the graded parts are: X4's
justification sentence, X5 parts three and four, X6's paper sentence, and X7's
defence. The numbers are the easy half everywhere.

In [1]:
# Path anchor and toolkit import. This is the only setup cell in the notebook.
#
# The bundle looks like this:
#     NEURIPS2026_EDU_GENERATE_VERIFY/
#         gv_toolkit.py          <- the shared toolkit
#         data/                 <- every cached file the exercises read
#         exercises/            <- you are here
#         solutions/
# so the root is one level up from this notebook, and data/ hangs off the root.
# Resolving it positionally means the same cell works locally, on Colab, and in a
# grader's batch run, with no absolute path anywhere.
import sys
from pathlib import Path

HERE = Path.cwd()
ROOT = HERE.parent if HERE.name in {"exercises", "solutions"} else HERE
sys.path.insert(0, str(ROOT))

import gv_toolkit as gv

DATA = gv.DATA
print("bundle root:", ROOT.name, "| cached inputs:", len(gv.REQUIRED_FILES) + len(gv.SUPPORTING_FILES))

bundle root: NEURIPS2026_EDU_GENERATE_VERIFY | cached inputs: 25


In [2]:
import json
import numpy as np
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import rdFingerprintGenerator

RDLogger.DisableLog("rdApp.*")

<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:34px 0 10px;color:#1A1416;"><span style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:13px;color:#A20025;font-weight:500;margin-right:12px;">&sect;4</span>X4. Where does the curve stop paying you</h2>

In [3]:
fm3 = gv.read_metrics_csv(DATA / "flowmol3_metrics.csv")
print(fm3.to_string(index=False))

by_steps = fm3.set_index("n_timesteps")
for K in [10, 25, 50, 100, 250]:
    row = by_steps.loc[K]
    print(f"\n{K:>3} steps: validity {row.validity_pct}% "
          f"({int(row.n_valid)} of {int(row.n_mols)}), mean SA {row.mean_sa}")

 n_timesteps  n_mols  n_valid  validity_pct  connected_pct  mean_qed  mean_sa
          10      12        5          41.7           75.0    0.5219   5.2277
          25      12       10          83.3           75.0    0.6230   3.5032
          50      12       12         100.0          100.0    0.7318   2.7947
         100      12       12         100.0          100.0    0.7231   2.4270
         250      12       12         100.0          100.0    0.6747   2.3181

 10 steps: validity 41.7% (5 of 12), mean SA 5.2277

 25 steps: validity 83.3% (10 of 12), mean SA 3.5032

 50 steps: validity 100.0% (12 of 12), mean SA 2.7947

100 steps: validity 100.0% (12 of 12), mean SA 2.427

250 steps: validity 100.0% (12 of 12), mean SA 2.3181


**The numbers.** Validity is 41.7 percent at 10 integration steps, 83.3 percent at
25, and 100 percent at 50, where it stays for 100 and 250. Validity saturates at 50.

Say those percentages with their denominator. The batch is twelve molecules at every
setting, so 41.7 percent is five of twelve and 83.3 percent is ten of twelve. A
single molecule flipping moves the headline figure by 8.3 points. Any percentage
computed on a denominator this small needs the denominator printed next to it, and a
learner who reports the percentages alone should be asked for it.

**The judgement.** No, 50 is not the answer to "how many steps do I need", and the
reason is in the next column. Mean synthetic accessibility, where lower means easier
for a chemist to make, keeps falling well past the point validity stops moving:
2.7947 at 50 steps, 2.4270 at 100, 2.3181 at 250. Validity has saturated and SA has
not.

So the knee is a property of the metric, not of the model. If chemical validity is
what you care about, 50 steps buys you everything and 250 buys you nothing. If
synthesizability is what you care about, 50 steps leaves value on the table. The
honest form of the answer is a question back: which metric is the budget for?

Check the `mean_qed` column as well: QED peaks at 50 steps (0.7318) and drops at
100 and 250 (0.7231, 0.6747). Three metrics on the same five-row table pick three
different step budgets because one scalar knob cannot optimize unaligned objectives
simultaneously.

<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:34px 0 10px;color:#1A1416;"><span style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:13px;color:#A20025;font-weight:500;margin-right:12px;">&sect;5</span>X5. The check that could not have failed</h2>

This is the centrepiece. The working is short; the discussion after it is not.

In [4]:
# Part one. The record census of the receptor file.
census = gv.pdb_record_census(DATA / "3rfm_protein.pdb")
print(census)
n_hetatm = census.get("HETATM", 0)
n_atom = census.get("ATOM", 0)
print(f"\nHETATM records: {n_hetatm}")
print(f"ATOM records:   {n_atom}")

# There is no REMARK or HEADER block: the file is bare coordinate records.
first = (DATA / "3rfm_protein.pdb").read_text().splitlines()[0]
print("\nfirst line of the file:", first[:60])

{'ATOM': 2250, 'TER': 1, 'END': 1}

HETATM records: 0
ATOM records:   2250

first line of the file: ATOM      1  N   SER A   7       6.159 -34.006 -51.651  1.00


In [5]:
# Part two. Closest approach of each generated ligand to the co-crystal caffeine.
caffeine = gv.read_sdf(DATA / "3rfm_caffeine_ref.sdf")[0]
ligands = gv.read_sdf(DATA / "diffsbdd_generated.sdf")
caffeine_xyz = caffeine.GetConformer().GetPositions()

closest_approach = []
for mol in ligands:
    xyz = mol.GetConformer().GetPositions()
    d = np.linalg.norm(xyz[:, None, :] - caffeine_xyz[None, :, :], axis=-1)
    closest_approach.append(float(d.min()))

median_closest = float(np.median(closest_approach))
max_closest = float(np.max(closest_approach))
print(f"closest approach across {len(ligands)} ligands")
print(f"  minimum {min(closest_approach):.3f} A")
print(f"  median  {median_closest:.3f} A")
print(f"  maximum {max_closest:.3f} A")
print(f"  all twenty within 0.6 A: {all(d < 0.6 for d in closest_approach)}")

# For contrast, the same measurement against the protein rather than the ligand.
protein_xyz = gv.pdb_atom_coords(DATA / "3rfm_protein.pdb")
lig0 = ligands[0].GetConformer().GetPositions()
d_prot = np.linalg.norm(lig0[:, None, :] - protein_xyz[None, :, :], axis=-1).min()
print(f"\nfor comparison, ligand 0's closest approach to any PROTEIN atom: "
      f"{d_prot:.2f} A")

closest approach across 20 ligands
  minimum 0.052 A
  median  0.346 A
  maximum 0.564 A
  all twenty within 0.6 A: True

for comparison, ligand 0's closest approach to any PROTEIN atom: 2.97 A


In [6]:
# Part three. The reported verdict, and what it would have been.
pb = gv.read_metrics_csv(DATA / "posebusters_diffsbdd.csv")
print("reported fully valid:", int(pb.pb_valid.sum()), "of", len(pb))
print("reported fully valid, treating unevaluable as failure:",
      int(pb.pb_valid_strict_na_as_fail.sum()), "of", len(pb))
print()
print("had the caffeine HETATM been retained, every ligand would have failed both "
      "organic-cofactor checks, so the fully valid count would have been 0 of 20.")
valid_if_caffeine_retained = 0

reported fully valid: 15 of 20
reported fully valid, treating unevaluable as failure: 13 of 20

had the caffeine HETATM been retained, every ligand would have failed both organic-cofactor checks, so the fully valid count would have been 0 of 20.


### The answer

**Part one.** Zero HETATM records. The file holds 2250 ATOM records, one TER and one
END, and nothing else. The PDB itself carries no REMARK or HEADER block; the
receptor preparation is recorded only in the comment header of
`posebusters_diffsbdd.csv`, which notes the full 2250-atom structure in place of a
36-residue pocket shell.

**Part two.** Every one of the twenty ligands has an atom within 0.564 angstroms of a
caffeine atom. The median closest approach is 0.346 angstroms and the minimum is
0.052. To put that in scale, a carbon-carbon single bond is about 1.54 angstroms and
a van der Waals contact between two carbons is around 3.4. These ligands are not near
the caffeine; they are interpenetrating it.

**Part three.** The two facts are not in tension once you see what the verifier was
given. `minimum_distance_to_organic_cofactors` and
`volume_overlap_with_organic_cofactors` both passed for all twenty ligands because
the receptor file contains no organic cofactors. There was nothing to be too close
to and nothing to overlap. The same holds for the inorganic-cofactor pair and the
water pair: six of the nine intermolecular checks in that table were structurally
incapable of returning False on this input.

Had the caffeine HETATM been left in the file, PoseBusters would have read it as an
organic cofactor, and every generated ligand overlaps it by construction, because
caffeine is the molecule whose binding site defined the pocket the generator was
conditioned on. Both organic-cofactor checks would have failed for all twenty and the
fully valid count would have dropped from 15 of 20 to 0 of 20.

**Part four.** Dropping the reference ligand is correct here. The design task is to
replace the caffeine, not to sit beside it, so a check that penalizes overlap with
the thing you are replacing is measuring the wrong thing.

It is not obvious, because the identical preparation would be wrong in the adjacent
case. Consider a heme-containing enzyme, a zinc metalloprotease, or any structure
where a catalytic metal or a bound cofactor is part of the functioning protein. Strip
those HETATM records and you get the same untroubled-looking table, except now the checks
that could not fail were the ones that should have caught your ligand sitting on top
of the catalytic zinc. The verifier cannot tell the two cases apart. Deciding which
bound molecules are scenery to be replaced and which are part of the target is a
scientific judgement about the biology, made by you, before the check runs.

### What you should now distrust

Three things, in increasing order of how much they will cost you.

**A pass rate reported without its input.** The 15 of 20 count in that table
reflects real intramolecular failures: 3 of 20 ligands failed bond angles, 2 failed
bond lengths, and 1 failed non-aromatic ring flatness (one ligand failed two
checks). On the intermolecular side, all twenty passed the three protein-distance
and protein-overlap checks, and those three evaluated passes sit in the same table,
in the same font, beside six cofactor and water checks whose 100 percent pass rate
carries zero information. Nothing in the output separates an evaluated pass from a
vacuous one, and `pb_valid` folds all of them into one boolean. PoseBusters
answered the exact question its inputs posed; the trap lies in reading a 20-check
summary as if all 20 checks had atoms to test.

**Treating a verification table as a statement about the model alone.** Every
verification result describes the pair of generated poses and verifier inputs.
Change the receptor preparation and the verdict flips from 15 of 20 to 0 of 20
without a single ligand atom moving. That dependence gives module 4 its title,
"Every check has a definition".

**Your own pipeline, most of all.** Nobody in the chain behaved badly here. The
receptor was prepared correctly for this task, by someone who knew why. The battery
ran correctly. The table was written out correctly. The vacuity emerges from the
combination, and it emerges silently, which means the only thing that catches it is
somebody asking what the verifier was given. In a pipeline you built
yourself, that somebody is you, and there is no error message to prompt you.

### The question to ask of any verification result you meet in a paper

One question, and it generalizes past this example:

**What input did the verifier receive, and which of its checks could that input have
caused to fail?**

Applied to a PoseBusters table in a 2026 paper, the follow-up questions are concrete.
Was the receptor stripped of HETATM records, and if so which ones. Were waters
retained. Was the pose evaluated against the full receptor or against a pocket shell,
and if a shell, how many residues. Was `dock` or `redock` configuration used, which
determines whether identity and RMSD checks were even in the list. Almost none of
these are usually stated, and each of them moves the number.

The habit transfers past PoseBusters. Applied to a docking benchmark: what was in the
decoy set, and could the model have distinguished it by molecular weight alone.
Applied to a retrosynthesis score: which reaction templates were in the library.
Applied to a held-out test split: was the split by molecule or by scaffold. In every
case the question is the same one, and in every case the answer is upstream of the
number rather than in it.

### A note on marking part four

Accept any answer that names a concrete case where retaining the bound molecule is
the correct preparation. A catalytic metal, a heme, a structural ion, a covalently
bound cofactor, or a second ligand in an allosteric site all qualify. Reject an answer
that says only "it depends on the target" without naming what it depends on, and
reject an answer that says the preparation used here was wrong, because it was not.

In [7]:
# The receptor's self-description lives in the PoseBusters provenance header rather
# than in the PDB file itself; the PDB is bare coordinate records.
pb_header = [ln for ln in (DATA / "posebusters_diffsbdd.csv").read_text().splitlines()
             if ln.startswith("#") and "2250" in ln]
for ln in pb_header:
    print(ln)

# X5 self-check values, for reference when marking.
print("HETATM", n_hetatm, "| ATOM", n_atom)
print("median closest approach", round(median_closest, 3),
      "| max", round(max_closest, 4))
print("reported valid", int(pb.pb_valid.sum()),
      "| valid if caffeine retained", valid_if_caffeine_retained)

# config='dock', mol_cond=assets/3rfm_protein.pdb (full 2250-atom receptor, NOT the 36-residue pocket shell)
HETATM 0 | ATOM 2250
median closest approach 0.346 | max 0.5643
reported valid 15 | valid if caffeine retained 0


<ol style="font-size:13.5px;color:#6B6461;padding-left:20px;">
<li style="margin:7px 0;">Buttenschoen M., Morris G. M., Deane C. M. (2024). PoseBusters: AI-based docking methods fail to generate physically valid poses or generalise to novel sequences. Chemical Science 15, 3130. <a href="https://doi.org/10.1039/d3sc04185a">doi:10.1039/d3sc04185a</a>. The failure you just found is the class of failure this paper is about: a generated pose that passes the checks that were run and would not pass the checks that were not.</li>
<li style="margin:7px 0;">Schneuing A. et al. (2024). Structure-based drug design with equivariant diffusion models. Nature Computational Science 4, 899. <a href="https://doi.org/10.1038/s43588-024-00737-x">doi:10.1038/s43588-024-00737-x</a>. The DiffSBDD model that produced the twenty ligands.</li>
</ol>

<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:34px 0 10px;color:#1A1416;"><span style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:13px;color:#A20025;font-weight:500;margin-right:12px;">&sect;6</span>X6. Two correct answers to one question</h2>

In [8]:
def diversity(mols, fingerprint):
    """1 minus the mean pairwise Tanimoto over all unordered pairs."""
    ms = [Chem.RemoveHs(m) for m in mols]          # trap 1: strip explicit Hs
    if fingerprint == "rdkit":
        fps = [Chem.RDKFingerprint(m) for m in ms]
    else:
        gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
        fps = [gen.GetFingerprint(m) for m in ms]
    sims = [DataStructs.TanimotoSimilarity(fps[i], fps[j])
            for i in range(len(fps)) for j in range(i + 1, len(fps))]
    return 1.0 - float(np.mean(sims)), len(sims)    # trap 2: mean over PAIRS


div_rdkit, n_pairs = diversity(ligands, "rdkit")
div_morgan, _ = diversity(ligands, "morgan")
print(f"RDKit topological (defaults):  {div_rdkit!r}")
print(f"Morgan r=2, 2048 bits:         {div_morgan!r}")
print(f"gap on the identical molecules: {abs(div_rdkit - div_morgan):.4f}")
print(f"pairs averaged over: {n_pairs} (not {len(ligands)})")

# The hydrogen convention, measured rather than asserted. The shipped SDF has no
# explicit hydrogens, so RemoveHs is a no-op on it; adding them is the real trap.
n_explicit_h = sum(1 for m in ligands for a in m.GetAtoms() if a.GetAtomicNum() == 1)
print(f"\nexplicit hydrogens in the shipped SDF: {n_explicit_h}")
div_asis, _ = diversity([m for m in ligands], "rdkit")
with_h = [Chem.AddHs(m, addCoords=True) for m in ligands]
fps_h = [Chem.RDKFingerprint(m) for m in with_h]
sims_h = [DataStructs.TanimotoSimilarity(fps_h[i], fps_h[j])
          for i in range(len(with_h)) for j in range(i + 1, len(with_h))]
print(f"with Chem.AddHs applied first: {1 - float(np.mean(sims_h))!r}")
ms = [Chem.RemoveHs(m) for m in ligands]
fps = [Chem.RDKFingerprint(m) for m in ms]
pairsum = sum(DataStructs.TanimotoSimilarity(fps[i], fps[j])
              for i in range(len(fps)) for j in range(i + 1, len(fps)))
print(f"dividing the pair sum by the molecule count instead: "
      f"{1 - pairsum / len(ms):.3f}  (not a diversity)")

RDKit topological (defaults):  0.8089776623453144
Morgan r=2, 2048 bits:         0.9175067835960684
gap on the identical molecules: 0.1085
pairs averaged over: 190 (not 20)

explicit hydrogens in the shipped SDF: 0
with Chem.AddHs applied first: 0.47766714002203947
dividing the pair sum by the molecule count instead: -0.815  (not a diversity)


**The numbers.** 0.8090 under the RDKit topological fingerprint at library defaults,
and 0.9175 under Morgan at radius 2 and 2048 bits. To full precision:
0.8089776623453144 and 0.9175067835960684. The gap is 0.1085 on the identical twenty
molecules.

**Neither is wrong.** The two fingerprints encode different things. The RDKit
topological fingerprint hashes linear paths through the molecular graph up to length
seven; Morgan hashes circular atom environments out to a radius of two bonds. They
disagree because they are asking different questions about similarity, and both
questions are reasonable. Reporting either one is defensible; reporting either one
unlabelled is not.

**The hydrogen trap, measured.** State the two cases separately.

The shipped `diffsbdd_generated.sdf` carries no explicit hydrogens. On that file
`Chem.RemoveHs` is a no-op and the value is byte-identical with and without it, as
the cell above prints.

What IS true, and much larger, is what happens when hydrogens are present. Add them
with `Chem.AddHs` and the RDKit topological diversity falls from 0.8090 to 0.4777,
and the Morgan value falls from 0.9175 to 0.8821. That is a swing of 0.33 on the
identical chemistry, larger than the gap between the two fingerprints. The lesson
survives intact and gets sharper: hydrogen handling is a definitional choice on the
same footing as the fingerprint, and a batch that reaches you with hydrogens attached
(from a strain calculation, from a docking pipeline, from a protonation step) will
give you a different diversity than the same batch without them.

**The denominator trap.** Dividing the sum of the 190 pair similarities by the number
of molecules rather than the number of pairs gives about minus 0.81, which is
impossible for one minus a Tanimoto mean. The arithmetic warns you if you look at it,
and the point of the trap is that people do not look.

One thing that does NOT matter, contrary to a common claim: whether you use
`np.mean` over the pair list or `sum()/len()` over the same list. Both agree exactly.
The decisive details are `RemoveHs` and the fingerprint choice.

**The sentence for a paper.** Something like: "Pairwise diversity, defined as one
minus the mean Tanimoto similarity over all 190 unordered pairs of the 20 generated
molecules, is 0.809 under the RDKit topological fingerprint at library defaults
(2048 bits, minPath 1, maxPath 7) after removal of explicit hydrogens."

That sentence is longer than the one most papers write, and a reader can reproduce
the number from it, which is the test. Accept any answer carrying all three of:
the fingerprint named with its parameters, the preparation step, and the pair
denominator. Reject a bare number, however correct. Refusing the bare number is the
objective.

<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:34px 0 10px;color:#1A1416;"><span style="font-family:ui-monospace,'SF Mono','JetBrains Mono',Menlo,monospace;font-size:13px;color:#A20025;font-weight:500;margin-right:12px;">&sect;7</span>X7. Sort four numbers by how much you can trust them</h2>

In [9]:
pc = gv.read_metrics_csv(DATA / "posecheck_diffsbdd.csv")
lig_rows = pc[~pc.is_reference]
ref_row = pc[pc.is_reference]
strain_median = float(lig_rows.strain_energy_kcal_mol.median())
strain_reference = float(ref_row.strain_energy_kcal_mol.iloc[0])
clash_median = float(lig_rows.clash_count.median())
cofactor_pass = float(pb["minimum_distance_to_organic_cofactors"].mean() * 100)

print(f"strain  median {strain_median:.1f} kcal/mol against a caffeine reference of "
      f"{strain_reference:.2f}, roughly {strain_median / strain_reference:.0f}x")
print(f"clash   median {clash_median:.1f} for the batch, "
      f"{float(ref_row.clash_count.iloc[0]):.0f} for the caffeine pose")
print(f"diversity {div_rdkit:.4f} under RDKit topological, "
      f"{div_morgan:.4f} under Morgan r=2")
print(f"cofactor pass rate {cofactor_pass:.0f} percent")

strain  median 134.5 kcal/mol against a caffeine reference of 0.39, roughly 347x
clash   median 2.5 for the batch, 3 for the caffeine pose
diversity 0.8090 under RDKit topological, 0.9175 under Morgan r=2
cofactor pass rate 100 percent


**The classifications.**

**Strain: drifts.** PoseCheck defines strain as the pose energy minus the energy of a
relaxed conformer ensemble, and it builds that ensemble with `AllChem.EmbedMolecule`
without passing a `randomSeed`. Re-running the batch therefore moves individual
ligands by up to a few percent while leaving most bit-identical. The median is about
134 kcal/mol against 0.39 for the caffeine co-crystal pose. The cell above prints that
ratio as 347, and the figure to write down is "over two orders of magnitude" instead
of any three-digit number: both numerator and denominator drift, making the ratio the
least stable quantity in the dataset. Never quote a strain number to more than
three significant figures and never present it as reproducible.

**Clash count: reproduces.** Median 2.5 for the batch and 3 for the caffeine
reference. It is computed from fixed input geometry with a fixed distance
tolerance and no embedding step, so the same input returns the same integer
every time.

Contrast those two PoseCheck columns: one tool call produces a drifting strain energy
alongside an exact clash count. Ask whether the specific metric is deterministic, and
check how it is computed instead of relying on the tool's name.

**Diversity: reproduces bit for bit, yet definition-dependent.** 0.8089776623453144
every time under the same fingerprint and preparation, and 0.9175 under Morgan r=2
with 2048 bits. Unlike strain, the computation itself is deterministic while the
upstream fingerprint choice shifts the value by 0.11. A reader who knows your
fingerprint reproduces your number bit for bit; a reader who does not cannot bound it.

**Cofactor pass rate: vacuous.** From X5. The receptor contains no organic cofactors,
so the check could only return 100 percent.

**A defensible ranking.** Clash count first, because it reproduces and its definition
is stated. Diversity second, provided the fingerprint is named, because it also
reproduces and one clause of prose fixes the definition. Strain third, quoted only as
an order of magnitude ("over two orders of magnitude above the co-crystal reference").
The cofactor pass rate last; omit it unless you state alongside it that the receptor
held zero cofactors.

Grade the defence over the exact ordering among the first three, with one hard rule:
the vacuous check must come last. Ranking a check that could not fail above a drifting
measurement confuses imprecision with zero information. More compute or a fixed seed
tightens an imprecise metric; only a different input fixes a vacuous one.

<hr style="border:0;border-top:1px solid #A20025;opacity:.55;margin:34px 0 12px;">
<h2 style="font-family:Georgia,'Iowan Old Style','Palatino Linotype',serif;font-weight:500;font-size:23px;margin:0 0 10px;color:#1A1416;">Answer summary</h2>

X4: validity 41.7, 83.3 and 100 percent at 10, 25 and 50 steps on a batch of twelve;
saturates at 50; but mean SA keeps falling to 2.4270 at 100 and 2.3181 at 250, so the
knee belongs to the metric rather than to the model.

X5: zero HETATM and 2250 ATOM records; every ligand within 0.564 angstroms of the
caffeine with a median closest approach of 0.346; six of nine intermolecular checks
could not have failed; the verdict would have been 0 of 20 rather than 15 of 20 had
the caffeine been retained; dropping a reference ligand you intend to replace is
right and dropping a catalytic metal is wrong, and no check can make that call.

X6: 0.8090 under RDKit topological and 0.9175 under Morgan r=2 2048, over 190 pairs,
with hydrogens absent. With explicit hydrogens attached the same two numbers are
0.4777 and 0.8821.

X7: strain drifts, clash count reproduces, diversity reproduces but is
definition-dependent, the cofactor pass rate is vacuous and belongs last.